In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import geopandas as gpd
import requests
from io import BytesIO

In [2]:


url = "https://minio.lab.sspcloud.fr/demon/velib/data/brute/velib-disponibilite-en-temps-reel.parquet"

response = requests.get(url)
response.raise_for_status()

with BytesIO(response.content) as buffer:
    df = gpd.read_parquet(buffer)

In [ ]:
df

#### Analyse exploratoire

In [ ]:
# On regarde les 5 premières lignes pour verifier que le fichier est chargé  
display(df.head())

# On vérifie le nom exact des colonnes et les types
print(df.info())

# On regarde s'il y a des valeurs manquantes
print(df.isna().sum())


Le jeu de donnée  Vélib Métropole comprend **1511 stations** et **15 colonnes**. 
- je constate qu'il n'y a aucune valeur manquante (`Null`) sur les colonnes principale de disponibilité sont : (`capacity`, `numbikesavailable`, `mechanical`, `ebike`).
- La colonne `station_opening_hours` est entièrement vide, nous n'en tiendrons pas compte pour la suite de notre étude

### Construction des indicateurs

#### Analyse du taux d'occupation globale

In [6]:
# On fait le calcul 
df['occupation_rate'] = df['numbikesavailable'] / df['capacity']
df['occupation_rate'] = df['occupation_rate'] * 100


df['occupation_rate'] = df['occupation_rate'].replace(np.inf, 0)

# On affiche la moyenne
print("Taux d'occupation moyen :")
print(df['occupation_rate'].mean())

Taux d'occupation moyen :
37.77153082181253


Le taux d'occupation moyen calculé sur l'ensemble du réseau est de **37,77 %**. 
Ce chiffre global montre que le réseau est globalement sein en fait : il reste de la place pour déposer des vélos (c'est à dire : pas de saturation à 100 %) et il y a suffisamment de vélos disponibles pour les usagers 

In [7]:
# On calcule le total de chaque type de vélo (electrique et mécanique)
nb_total_electrique = df['ebike'].sum()
nb_total_mecanique = df['mechanical'].sum()
n = df['numbikesavailable'].sum() #avec n le nombre de vélos total

# On calcule le pourcentage de vélos électriques
# (Nombre électrique / n) * 100
part_electrique = (nb_total_electrique / n) * 100

# On calcul le pourcentage de vélos méca
# (nombre de velos mécanique / n) * 100
part_mecanique = (nb_total_mecanique / n) * 100

print(f"Total vélos électriques : {nb_total_electrique}")
print(f"Total vélos mécaniques : {nb_total_mecanique}")
print(f"La part de l'électrique est de : {part_electrique:.2f}% et la part de vélo mécanique est de : {part_mecanique:.2f}%")


Total vélos électriques : 7495
Total vélos mécaniques : 11328
La part de l'électrique est de : 39.82% et la part de vélo mécanique est de : 60.18%


### Analyse par commune 

In [ ]:
# ici je vais faire un groupby des données par le nom de la commune
# Et lui demandé la moyenne pour le nombre de vélos et le taux d'occupation
commune = df.groupby('nom_arrondissement_communes')[['numbikesavailable', 'occupation_rate']].mean()

#print(commune)
# je trie tableau pour avoir les communes avec le plus de vélos en premier
commune = commune.sort_values(by='numbikesavailable', ascending=False)

# top 10 premières des communes 
print("--- TOP 10 DES COMMUNES PAR DISPONIBILITÉ ---")
display(commune.head(10))

### Graphique

In [ ]:
# On configure un style pour les graphique
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [ ]:

nb_meca_secu = df['mechanical'].sum()
nb_elec_secu = df['ebike'].sum()

# Configuration des données du graphique
sizes = [nb_meca_secu, nb_elec_secu]
labels = ['Vélos Mécaniques', 'Vélos Électriques']
colors = ['#1f77b4', '#ff7f0e']

plt.figure(figsize=(6, 5))
plt.pie(sizes, labels=labels, startangle=90, colors=colors)
plt.title("Répartition globale des types de vélos disponibles")
plt.axis('equal')
plt.show()

Le graphique en secteur met en évidence une répartition clair au sein du réseau :
- **Vélos mécaniques (classiques) :** ~60,18 % de la flotte disponible.
- **Vélos électriques :** ~39,82 % de la flotte disponible.

### graphique combiné

In [ ]:
# on recalcule 'occupation_rate' ici pour être sûr qu'il existe dans df
df['occupation_rate'] = (df['numbikesavailable'] / df['capacity']) * 100

# On fait le groupby et le tri
commune = df.groupby('nom_arrondissement_communes')[['numbikesavailable', 'occupation_rate']].mean()
commune = commune.sort_values(by='numbikesavailable', ascending=False)

# On isole le Top 5 pour le graphique
top5_data = commune.head(5).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 6))

# Histogramme pour la disponibilité moyenne
color_bar = '#34495e'
ax1.set_xlabel('Communes')
ax1.set_ylabel('Nombre moyen de vélos disponibles', color=color_bar)
ax1.bar(top5_data['nom_arrondissement_communes'], top5_data['numbikesavailable'], color=color_bar, alpha=0.8, width=0.5)
ax1.tick_params(axis='y', labelcolor=color_bar)

# Deuxième axe (Axe vertical droit pour le taux d'occupation)
ax2 = ax1.twinx()
color_line = '#e74c3c'
ax2.set_ylabel("Taux d'occupation moyen (%)", color=color_line)
ax2.plot(top5_data['nom_arrondissement_communes'], top5_data['occupation_rate'], color=color_line, marker='o', linewidth=2.5)
ax2.tick_params(axis='y', labelcolor=color_line)

plt.title("Top 5 des communes : Volume disponible et Taux d'occupation")
fig.tight_layout()
plt.show()